# Siamese Facial Recognition — Train on Kaggle (GPU)

**From-scratch ResNet-18-style Siamese network + Triplet (semi-hard) loss.**

This notebook is designed to run **directly on Kaggle** with the free **T4 x2 GPU**.
It is built for **few-shot** data: only 2–3 photos per person in
`Photos_dataset/<person_name>/*.jpg`.

### Accuracy-maximizing techniques used here
1. **MTCNN face detection + 5-point landmark alignment** to a canonical 112×112
   template (the single biggest accuracy lever for few-shot).
2. **Heavy augmentation** (torchvision) → each 2–3 photo person yields dozens
   of training samples (flip, affine, brightness/contrast, blur, Gaussian noise,
   cutout/random-erasing).
3. **ResNet-18-style backbone (from scratch) + Squeeze-and-Excitation** channel
   attention, 512-d **L2-normalized** embeddings.
4. **Triplet loss with semi-hard negative mining** (negative pool refreshed each
   epoch from current model embeddings).
5. **Mixed-precision (AMP)**, **cosine LR schedule with warmup**, **EMA weights**.
6. **ROC-calibrated decision threshold** exported to `threshold.json`.
7. **Prototypical enrollment** at inference (mean of multiple embeddings/person).

### Robustness
- **Cell 1 self-heals the environment**: it probes torch in a subprocess and, if
  the GPU kernels don't match (`cudaErrorNoKernelImageForDevice`), reinstalls a
  compatible PyTorch automatically — **no kernel restart needed**.
- **Face detection runs on CPU** (one-time step), so a GPU issue can never block
  preprocessing; training still uses the GPU.

### How to run
1. Zip your local `Photos_dataset/` folder and create a **Kaggle Dataset** from it
   (see `kaggle_setup.md`). The dataset must contain one sub-folder per person.
2. Attach that dataset to this notebook (**Add Input → Your Dataset**).
3. **Settings → Internet = ON**, **Accelerator = GPU T4 x2**.
4. Run All. At the end, download `siamese_backbone.pth` and `threshold.json`
   from the Output panel and drop them into the project's `models/` folder.


In [ ]:
# Cell 1 — environment SELF-HEAL + install extras.
# This cell must run BEFORE any 'import torch' in the notebook. We probe torch
# inside a SUBPROCESS so we test the actually-installed binary WITHOUT loading a
# stale torch into this kernel. That means a repair needs NO kernel restart:
# the freshly pip-installed torch is first imported by Cell 2.
import sys, subprocess, importlib

def _probe_torch():
    code = (
        "import torch\n"
        "ca = torch.cuda.is_available()\n"
        "print('VERSION', torch.__version__)\n"
        "print('CUDA', torch.version.cuda)\n"
        "print('CUDA_AVAIL', ca)\n"
        "if ca:\n"
        "    print('DEVICE', torch.cuda.get_device_name(0))\n"
        "    print('CAP', torch.cuda.get_device_capability(0))\n"
        "    x = torch.randn(64, 64, device='cuda')\n"
        "    _ = (x @ x).sum().item()\n"
        "    print('GPU_OK')\n"
    )
    return subprocess.run([sys.executable, "-c", code],
                          capture_output=True, text=True)

r = _probe_torch()
print(r.stdout.strip())
if r.stderr.strip():
    print("STDERR (tail):", r.stderr.strip()[-300:])

cuda_avail = "CUDA_AVAIL True" in r.stdout
gpu_ok = "GPU_OK" in r.stdout

if cuda_avail and not gpu_ok:
    # torch is a CUDA build but its kernels don't match the GPU's compute
    # capability (the 'no kernel image is available' error). Reinstall a
    # broadly-compatible build: cu121 wheels cover T4(sm75), P100(sm60),
    # L4(sm89), Ampere(sm80/86), H100(sm90).
    print(">> GPU kernel mismatch detected -> reinstalling torch 2.5.1 / cu121 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "torch==2.5.1", "torchvision==0.20.1",
        "--index-url", "https://download.pytorch.org/whl/cu121"])
    r2 = _probe_torch()
    print(r2.stdout.strip())
    if "GPU_OK" not in r2.stdout:
        raise RuntimeError(
            "GPU still unusable after reinstall. Open Settings -> Accelerator "
            "and switch GPU type, or run with no accelerator (CPU, slower).")
    print(">> Repaired. Continuing (no restart needed).")
elif not cuda_avail:
    print(">> No CUDA GPU detected. Enable Settings -> Accelerator -> GPU T4 x2 "
          "for speed. Training will otherwise run on CPU.")

# Install facenet-pytorch WITHOUT touching torch/numpy/Pillow (its real deps are
# already present) to avoid the dependency-downgrade cascade that broke
# torchvision + PIL earlier.
try:
    importlib.import_module("facenet_pytorch")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--no-deps", "facenet-pytorch"])
print(">> Environment ready.")


In [ ]:
# Cell 2 — imports & global config
import os, json, random, math, copy, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import (roc_curve, precision_recall_curve,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
import matplotlib.pyplot as plt

# torchvision transforms for augmentation (stable, already on Kaggle, numpy-2 safe).
import torchvision.transforms as T

from facenet_pytorch import MTCNN

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

# ----------------------------------------------------------------------------
@dataclass
class Cfg:
    # --- Paths ---------------------------------------------------------------
    # >>> CHANGE THIS to your attached Kaggle dataset mount point if needed <<<
    dataset_root: str = "/kaggle/input/photos-dataset"          # contains <name>/*.jpg
    output_root:  str = "/kaggle/working"                       # where artifacts are saved

    # --- Model / image -------------------------------------------------------
    img_size: int = 112
    embedding_dim: int = 512
    dropout: float = 0.4

    # --- Augmentation --------------------------------------------------------
    aug_per_crop: int = 12        # extra augmented copies generated per crop (precompute)

    # --- Training ------------------------------------------------------------
    epochs: int = 60               # cap (5-20 people); early stopping ends it sooner
    batch_size: int = 64           # triplets per batch
    lr: float = 1e-3
    weight_decay: float = 1e-4
    triplet_margin: float = 0.5    # euclidean margin on L2-normalized embeddings (FaceNet default)
    hard_neg_k: int = 32           # negatives sampled, hardest-within-margin chosen
    temperature: float = 1.0
    ema_decay: float = 0.9995      # smoother EMA weights
    warmup_epochs: int = 3         # warmup for cosine LR schedule
    patience: int = 7              # early-stop if val F1 stalls for this many epochs
    min_epochs: int = 10           # never stop before this
    seed: int = 42

    # --- Data split ----------------------------------------------------------
    val_holdout: float = 0.25      # fraction of CROPS held out per person for val pairs
    min_crops_after_split: int = 1 # keep at least this many train crops per person

CFG = Cfg()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def set_seed(s=CFG.seed):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed()


In [ ]:
# Cell 3 — discover people & images (with auto-detection)
_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _person_folders(root: Path) -> Dict[str, List[Path]]:
    """Return {person_name: [image paths]} for child dirs of `root` that hold images."""
    out = {}
    if not root.exists():
        return out
    for p in sorted(root.iterdir()):
        if not p.is_dir():
            continue
        files = [f for f in sorted(p.iterdir()) if f.is_file() and f.suffix.lower() in _EXTS]
        if files:
            out[p.name] = files
    return out

def auto_find_dataset(min_people: int = 2) -> Path:
    """Walk /kaggle/input to locate a dir whose children are image-bearing
    'person' folders. Makes the notebook immune to mount-path / zip-nesting
    differences (e.g. /kaggle/input/photos-dataset/Photos_dataset/<person>)."""
    best, best_n = None, 0
    roots = [Path("/kaggle/input"), Path("./input"), Path(".")]
    for base in roots:
        if not base.exists():
            continue
        for d in [base] + [p for p in base.rglob("*") if p.is_dir()]:
            pf = _person_folders(d)
            if len(pf) >= min_people and len(pf) > best_n:
                best, best_n = d, len(pf)
    return best

def discover_dataset(root: str) -> Dict[str, List[Path]]:
    persons = _person_folders(Path(root))
    if not persons:
        # Auto-detect instead of failing — the configured path is often wrong
        # (website-URL shape vs the real /kaggle/input/<slug> mount).
        found = auto_find_dataset()
        if found is not None:
            print(f"Configured root '{root}' had no person folders; "
                  f"auto-detected dataset at: {found}")
            persons = _person_folders(found)
            CFG.dataset_root = str(found)
        else:
            raise SystemExit(
                "Could not find a dataset of person folders under /kaggle/input.\n"
                "Attach your dataset (Add Input) and check its mount path with:\n"
                "    !ls /kaggle/input")
    print(f"Found {len(persons)} people, "
          f"{sum(len(v) for v in persons.values())} images total. "
          f"(root: {CFG.dataset_root})")
    for name, files in list(persons.items())[:10]:
        print(f"  - {name}: {len(files)} img")
    return persons

PERSONS = discover_dataset(CFG.dataset_root)


In [ ]:
# Cell 4 — MTCNN face detection + 5-point alignment (canonical 112x112)
# Canonical ArcFace template (where each landmark should land).
ARCFACE_TEMPLATE = np.array([
    [38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
    [41.5493, 92.3655], [70.7299, 92.2041],
], dtype=np.float32)

def make_mtcnn() -> MTCNN:
    # Run face detection on CPU deliberately: it is a one-time preprocessing step
    # and this keeps it immune to any GPU/CUDA issue (e.g. kernel-image mismatch).
    # Training (the heavy part) still uses the GPU.
    return MTCNN(image_size=Cfg.img_size, margin=0, min_face_size=40,
                 thresholds=[0.6, 0.7, 0.7], factor=0.709,
                 post_process=False, keep_all=False, device="cpu")

import cv2  # OpenCV is always present on Kaggle; used for the affine warp below.

def align_face(img_pil, landmarks):
    """Affine-warp `landmarks` (5x2) onto ARCFACE_TEMPLATE -> 112x112 RGB uint8.

    Uses cv2.warpAffine (the FaceNet/InsightFace standard). We deliberately do
    NOT use skimage.warp here: current skimage only treats a matrix as a
    transform if it is 3x3, and a 2x3 array is misread as a coordinate array,
    raising 'invalid shape for coordinate array'."""
    lm = np.asarray(landmarks, dtype=np.float32).reshape(5, 2)
    arr = np.array(img_pil.convert("RGB"))
    M, _ = cv2.estimateAffinePartial2D(lm, ARCFACE_TEMPLATE)
    if M is None:                       # degenerate landmarks -> identity crop
        M = np.eye(2, 3, dtype=np.float32)
    return cv2.warpAffine(arr, M, (Cfg.img_size, Cfg.img_size),
                          flags=cv2.INTER_LINEAR, borderValue=0)

@torch.no_grad()
def detect_and_align(path, mtcnn):
    img = Image.open(path).convert("RGB")
    boxes, probs, landmarks = mtcnn.detect(img, landmarks=True)
    if boxes is None or len(boxes) == 0:
        return None
    # pick largest face
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    idx = int(np.argmax(areas))
    return align_face(img, landmarks[idx])

# Build aligned crop cache: {person_name: [np.uint8 (H,W,3), ...]}
print("Detecting & aligning faces (this runs once)...")
mtcnn = make_mtcnn()
ALIGNED: Dict[str, List[np.ndarray]] = {}
missing = 0
for name, files in PERSONS.items():
    crops = []
    for f in files:
        crop = detect_and_align(f, mtcnn)
        if crop is None:
            missing += 1
            continue
        crops.append(crop)
    if crops:
        ALIGNED[name] = crops
print(f"Aligned faces for {len(ALIGNED)} people. Skipped {missing} images (no face).")

# Quick montage preview
fig, axs = plt.subplots(2, 6, figsize=(12, 4))
for ax in axs.ravel(): ax.axis("off")
shown = 0
for name, crops in ALIGNED.items():
    for crop in crops:
        if shown >= axs.size: break
        axs.ravel()[shown].imshow(crops[0]); axs.ravel()[shown].set_title(name, fontsize=8)
        shown += 1
    if shown >= axs.size: break
plt.tight_layout(); plt.show()


In [ ]:
# Cell 5 — augmentation via torchvision (stable, no extra dependency, numpy-2 safe).
# Heavy augmentation is what makes 2-3 images/person trainable.
# We intentionally avoided albumentations: its newer versions changed transform
# APIs (e.g. GaussNoise) and pulled in dependency downgrades on Kaggle.
_MEAN = (0.485, 0.456, 0.406)
_STD  = (0.229, 0.224, 0.225)
IMG = Cfg.img_size

class _AddGaussianNoise:
    """torchvision has no built-in additive noise; tiny custom tensor transform."""
    def __init__(self, std=0.05, p=0.3):
        self.std, self.p = std, p
    def __call__(self, x):
        if torch.rand(1).item() < self.p:
            x = x + torch.randn_like(x) * self.std
        return x

def _aug_compose():
    # Rebuilt per call so each sample gets a fresh random draw.
    return T.Compose([
        T.ToPILImage(),                                            # HWC uint8 -> PIL
        T.Resize((IMG, IMG)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAffine(degrees=15, translate=(0.05, 0.05),
                       scale=(0.9, 1.1), fill=0),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.ToTensor(),
        _AddGaussianNoise(std=0.05, p=0.3),
        T.Normalize(_MEAN, _STD),
        T.RandomErasing(p=0.4, scale=(0.02, 0.15), value=0),       # cutout
    ])

EVAL_TFM = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG, IMG)),
    T.ToTensor(),
    T.Normalize(_MEAN, _STD),
])

def to_aug_tensor(crop_uint8, rng=None):
    return _aug_compose()(crop_uint8)            # CHW float tensor

def to_eval_tensor(crop_uint8):
    return EVAL_TFM(crop_uint8)


In [ ]:
# Cell 6 — train/val split by CROPS (so val pairs are unseen faces-of-same-person)
def split_crops(aligned, holdout, min_train):
    train, val = {}, {}
    rng = random.Random(Cfg.seed)
    for name, crops in aligned.items():
        n = len(crops)
        idx = list(range(n)); rng.shuffle(idx)
        n_val = 1 if n <= 2 else max(1, int(round(n * holdout)))
        n_val = min(n_val, n - min_train)
        val_idx = set(idx[:n_val]); tr_idx = [i for i in idx if i not in val_idx]
        train[name] = [crops[i] for i in tr_idx]
        val[name]   = [crops[i] for i in val_idx]
    # drop people with zero train crops (edge case)
    train = {k: v for k, v in train.items() if len(v) >= 1}
    return train, val

TRAIN_CROPS, VAL_CROPS = split_crops(ALIGNED, Cfg.val_holdout, Cfg.min_crops_after_split)
print(f"Train people: {len(TRAIN_CROPS)} | "
      f"Val people: {len(VAL_CROPS)} | "
      f"train crops: {sum(len(v) for v in TRAIN_CROPS.values())} | "
      f"val crops: {sum(len(v) for v in VAL_CROPS.values())}")


In [ ]:
# Cell 7 — MODEL DEFINITION  (must match app/model.py EXACTLY for checkpoint loading)
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_ch, out_ch, stride=1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch, reduction=reduction)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = out + identity
        return F.relu(out, inplace=True)

class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=Cfg.embedding_dim, dropout=Cfg.dropout):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(512, embedding_dim))
        self._init_weights()
    @staticmethod
    def _make_layer(in_ch, out_ch, blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride=stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x); x = self.head(x)
        return F.normalize(x, p=2, dim=1)

net = EmbeddingNet().to(DEVICE)
n_params = sum(p.numel() for p in net.parameters())
print(f"EmbeddingNet parameters: {n_params:,} (~{n_params/1e6:.2f}M)")


In [ ]:
# Cell 8 — TripletFaceDataset with semi-hard negative mining
class TripletFaceDataset(Dataset):
    """
    Each item returns an (anchor, positive, negative) triplet of augmented tensors.

    Negative mining: we cache each crop's clean embedding under the current model
    (refresh_epoch). When sampling a negative we draw `hard_neg_k` candidate crops
    from OTHER people and pick the one whose cached distance to the anchor is the
    SMALLEST that still exceeds the anchor-positive distance (semi-hard). Falls
    back to the hardest overall if none qualifies. This keeps training stable.
    """
    def __init__(self, crops_by_person: Dict[str, List[np.ndarray]],
                 steps_per_epoch: int = 4096, hard_neg_k: int = Cfg.hard_neg_k):
        self.names = list(crops_by_person.keys())
        self.crops = crops_by_person
        self.steps = steps_per_epoch
        self.k = hard_neg_k
        # flatten crops with person index
        self.flat: List[Tuple[int, np.ndarray]] = []
        for pi, name in enumerate(self.names):
            for crop in self.crops[name]:
                self.flat.append((pi, crop))
        self.emb_cache = None  # np.ndarray (N, D) in crop order

    def num_crops(self):
        return len(self.flat)

    @torch.no_grad()
    def refresh_cache(self, model):
        model.eval()
        embs = []
        bs = 128
        with torch.no_grad():
            for i in range(0, len(self.flat), bs):
                batch = torch.stack(
                    [to_eval_tensor(c) for _, c in self.flat[i:i+bs]]).to(DEVICE)
                e = model(batch).cpu().numpy()
                embs.append(e)
        self.emb_cache = np.concatenate(embs, axis=0).astype(np.float32)
        self.cache_person = np.array([pi for pi, _ in self.flat])
        model.train()

    def __len__(self):
        return self.steps

    def _aug(self, crop):
        return to_aug_tensor(crop)

    def __getitem__(self, idx):
        n = len(self.flat)
        ai = random.randint(0, n - 1)
        a_pi, a_crop = self.flat[ai]

        # positive: different crop of same person if available, else augmented anchor
        same_pool = [j for j in range(n) if self.flat[j][0] == a_pi and j != ai]
        if same_pool:
            pi_crop = self.flat[random.choice(same_pool)][1]
            pos_crop = pi_crop
        else:
            pos_crop = a_crop

        anchor = self._aug(a_crop)
        positive = self._aug(pos_crop)

        # negative: semi-hard mining using cache
        negative = self._mine_negative(ai, a_pi)

        return anchor, positive, negative

    def _mine_negative(self, ai, a_pi):
        if self.emb_cache is None:
            # random negative (first epoch before cache exists)
            while True:
                j = random.randint(0, len(self.flat) - 1)
                if self.flat[j][0] != a_pi:
                    return self._aug(self.flat[j][1])
        a_emb = self.emb_cache[ai]
        cand = np.random.randint(0, len(self.flat), size=self.k)
        cand = cand[self.cache_person[cand] != a_pi]
        if len(cand) == 0:
            j = random.randint(0, len(self.flat) - 1)
            return self._aug(self.flat[j][1])
        d = self.emb_cache[cand] @ a_emb          # cosine sim (normalized)
        # hardest = smallest cosine sim (largest distance)
        order = np.argsort(d)                      # ascending similarity
        # prefer semi-hard: among candidates, choose one not *too* hard sometimes
        # to avoid collapse. We pick the median-hardest with 50% prob for stability.
        if random.random() < 0.5:
            pick = cand[order[len(order)//2]]
        else:
            pick = cand[order[0]]
        return self._aug(self.flat[pick][1])

train_ds = TripletFaceDataset(TRAIN_CROPS, steps_per_epoch=4096)
# Faster data pipeline: more workers + persistent workers + prefetch so the
# augmentation CPU work overlaps with the GPU and doesn't starve it.
train_loader = DataLoader(train_ds, batch_size=Cfg.batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=4)
print(f"Triplet steps/epoch: {len(train_ds)} | batches: {len(train_loader)}")


In [ ]:
# Cell 9 — loss, optimizer, scheduler, EMA, AMP
criterion = nn.TripletMarginLoss(margin=Cfg.triplet_margin, p=2)

# Param groups: no weight decay on BN/bias
decay, no_decay = [], []
for n, p in net.named_parameters():
    if not p.requires_grad: continue
    (no_decay if (n.endswith(".bias") or "bn" in n) else decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": Cfg.weight_decay},
     {"params": no_decay, "weight_decay": 0.0}], lr=Cfg.lr)

def cosine_warmup(step, total, warmup_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

total_steps = Cfg.epochs * len(train_loader)
warmup_steps = Cfg.warmup_epochs * len(train_loader)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: cosine_warmup(s, total_steps, warmup_steps))

# AMP scaler (torch 2.x new API). Disabled (no-op) on CPU.
use_amp = (DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# EMA weights
ema_net = copy.deepcopy(net).eval()
for p in ema_net.parameters(): p.requires_grad_(False)
@torch.no_grad()
def update_ema(model, ema, decay):
    for ep, p in zip(ema.parameters(), model.parameters()):
        ep.mul_(decay).add_(p.detach(), alpha=1 - decay)
    for eb, b in zip(ema.buffers(), model.buffers()):
        eb.copy_(b)


In [ ]:
# Cell 10 — evaluation helpers: genuine/impostor pairs + ROC threshold
def all_embeddings(model, crops_by_person, tta=0):
    """Return dict {name: list_of_embedding_arrays} using clean eval transform."""
    model.eval()
    out = {}
    with torch.no_grad():
        for name, crops in crops_by_person.items():
            embs = []
            for c in crops:
                t = to_eval_tensor(c).unsqueeze(0).to(DEVICE)
                e = model(t).cpu().numpy()[0]
                embs.append(e)
            out[name] = embs
    model.train()
    return out

def build_pairs(emb_by_person):
    """genuine (same person) & impostor (diff person) cosine distances."""
    gen, imp = [], []
    names = list(emb_by_person.keys())
    for i, n in enumerate(names):
        E = emb_by_person[n]
        for a in range(len(E)):
            for b in range(a+1, len(E)):
                gen.append(float(1.0 - np.dot(E[a], E[b])))
        for j in range(i+1, len(names)):
            for a in E:
                for b in emb_by_person[names[j]]:
                    imp.append(float(1.0 - np.dot(a, b)))
    return np.array(gen), np.array(imp)

def best_threshold_roc(gen, imp):
    """Cosine-distance threshold maximizing Youden's J on the val pairs."""
    y_true = np.concatenate([np.ones_like(gen), np.zeros_like(imp)])
    y_dist = np.concatenate([gen, imp])           # small => genuine
    fpr, tpr, thr = roc_curve(y_true, -y_dist)    # negate so higher==genuine
    j = tpr - fpr
    best = int(np.argmax(j))
    # thr here is on -distance; convert back to distance threshold
    return float(-thr[best])

def metrics_at(dist_same, dist_diff, thr):
    y_true = np.concatenate([np.ones_like(dist_same), np.zeros_like(dist_diff)])
    y_pred = (np.concatenate([dist_same, dist_diff]) <= thr).astype(int)
    return {
        "threshold": float(thr),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": float(roc_auc_score(y_true, -np.concatenate([dist_same, dist_diff]))),
    }

def loo_eval(model, crops_by_person):
    """Leave-one-out evaluation (robust for TINY datasets).

    Each crop is a query; the gallery = all OTHER crops.
      genuine dists  = query vs same-person other crops
      impostor dists = query vs different-person crops
    This produces real pairs even when only 1 crop/person is held out, so val
    metrics are never artificially 0 (the old build_pairs fallback).
    For very small data the gallery necessarily includes train crops (slightly
    optimistic) -- the only honest option with <5 people.
    Returns (gen, imp, metrics_or_None, threshold)."""
    emb = all_embeddings(model, crops_by_person)
    items = [(e, name) for name, es in emb.items() for e in es]
    gen, imp = [], []
    for i, (qe, qname) in enumerate(items):
        for j, (oe, oname) in enumerate(items):
            if i == j:
                continue
            d = float(1.0 - np.dot(qe, oe))
            (gen if oname == qname else imp).append(d)
    gen, imp = np.array(gen), np.array(imp)
    if len(gen) == 0 or len(imp) == 0:
        return gen, imp, None, Cfg.triplet_margin
    thr = best_threshold_roc(gen, imp)
    return gen, imp, metrics_at(gen, imp, thr), thr


In [ ]:
# Cell 11 — training loop (leave-one-out validation)
if len(ALIGNED) < 5:
    print("!! NOTE: %d people enrolled. This pipeline targets 5-20 people (2-3 "
          "photos each) for a usable embedding space. With fewer identities the "
          "model separates the known faces but generalizes poorly to new ones.\n"
          % len(ALIGNED))
else:
    print("Dataset: %d people, %d crops. Training a 5-20 identity few-shot "
          "Siamese embedding space.\n" % (len(ALIGNED), sum(len(v) for v in ALIGNED.values())))

history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": [], "val_thr": []}
best_f1 = -1.0
best_state = None
best_thr = Cfg.triplet_margin
best_epoch = 0
epochs_no_improve = 0
stopped_early = False

# initial cache
train_ds.refresh_cache(net)

import time
for epoch in range(1, Cfg.epochs + 1):
    t0 = time.time()
    net.train()
    running = 0.0
    for a, p, n in train_loader:
        a, p, n = a.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            ea, ep, en = net(a), net(p), net(n)
            loss = criterion(ea, ep, en)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        running += loss.item()
    update_ema(net, ema_net, Cfg.ema_decay)

    avg_loss = running / len(train_loader)
    history["train_loss"].append(avg_loss)

    # ---- leave-one-out validation over ALL crops (tiny-data-safe) ----
    ema_net.eval()
    gen, imp, m, thr = loo_eval(ema_net, ALIGNED)
    if m is None:
        m = {"accuracy":0, "f1":0, "precision":0, "recall":0, "auc":0, "threshold":thr}
    history["val_thr"].append(thr)
    history["val_f1"].append(m["f1"]); history["val_acc"].append(m["accuracy"])
    history["val_loss"].append(avg_loss)

    dt = time.time() - t0
    eta = dt * (Cfg.epochs - epoch)
    star = "*" if m["f1"] > best_f1 + 1e-4 else " "
    print(f"E{epoch:02d}{star}| loss {avg_loss:.4f} | "
          f"acc {m['accuracy']:.3f} | f1 {m['f1']:.3f} | "
          f"thr {thr:.3f} | auc {m['auc']:.3f} | "
          f"lr {scheduler.get_last_lr()[0]:.2e} | {dt:.1f}s (eta {eta/60:.1f}m)")

    # ---- best-checkpoint + early stopping ----
    if m["f1"] > best_f1 + 1e-4:
        best_f1 = m["f1"]; best_thr = thr
        best_state = copy.deepcopy(ema_net.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epoch >= Cfg.min_epochs and epochs_no_improve >= Cfg.patience:
            stopped_early = True
            print(f">> Early stopping at epoch {epoch}: F1 stalled for "
                  f"{Cfg.patience} epochs (best F1={best_f1:.4f} @ epoch {best_epoch}).")
            break

    # refresh triplet cache for next epoch with current (EMA) weights
    train_ds.refresh_cache(ema_net)

if best_state is None:           # safety net if training produced nothing usable
    best_state = copy.deepcopy(ema_net.state_dict())
tag = "EARLY-STOPPED" if stopped_early else "completed"
print(f"\nTraining {tag}. Best F1: {best_f1:.4f} @ epoch {best_epoch} (threshold {best_thr:.4f})")


In [ ]:
# Cell 12 — training curves
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].plot(history["train_loss"]); axs[0].set_title("Train triplet loss"); axs[0].set_xlabel("epoch")
axs[1].plot(history["val_acc"], label="acc"); axs[1].plot(history["val_f1"], label="f1")
axs[1].set_title("Validation metrics"); axs[1].set_xlabel("epoch"); axs[1].legend()
axs[2].plot(history["val_thr"]); axs[2].set_title("Calibrated threshold (cos dist)"); axs[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()


In [ ]:
# Cell 13 — final evaluation with best EMA weights + ROC/PR curves
net.load_state_dict(best_state)
net.eval(); ema_net.load_state_dict(best_state); ema_net.eval()

# Final evaluation via leave-one-out over ALL crops. (A separate held-out-crop
# "validation" report is omitted: with only 2-3 photos/person the per-person
# holdout always yields 1 crop -> zero genuine pairs -> uninformative. LOO over
# all crops is the honest, real-metric evaluation for this few-shot regime.)
def report(crops_by_person, tag):
    """Final evaluation via leave-one-out (real metrics at any size)."""
    gen, imp, m, thr = loo_eval(ema_net, crops_by_person)
    if m is None:
        print(f"[{tag}] not enough pairs"); return None
    print(f"[{tag}] thr={thr:.3f} acc={m['accuracy']:.3f} prec={m['precision']:.3f} "
          f"rec={m['recall']:.3f} f1={m['f1']:.3f} auc={m['auc']:.3f} "
          f"(gen={len(gen)}, imp={len(imp)})")

    fpr, tpr, _ = roc_curve(np.concatenate([np.ones_like(gen), np.zeros_like(imp)]),
                            -np.concatenate([gen, imp]))
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC={m['auc']:.3f}")
    plt.plot([0,1],[0,1],"--", alpha=0.4)
    plt.title(f"ROC — {tag}")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.show()
    return m

m_all = report(ALIGNED, "leave-one-out (all crops)")
m_val = m_all   # alias so Cell 15's threshold logic stays valid

# Distance distribution (leave-one-out distances)
gen, imp, _, _ = loo_eval(ema_net, ALIGNED)
plt.figure(figsize=(6,4))
plt.hist(gen, bins=40, alpha=0.6, label="genuine")
plt.hist(imp, bins=40, alpha=0.6, label="impostor")
plt.axvline(best_thr, color="k", ls="--", label=f"thr={best_thr:.3f}")
plt.xlabel("cosine distance"); plt.legend(); plt.title("Genuine vs impostor (LOO)"); plt.show()


In [ ]:
# Cell 14 — t-SNE of embeddings (visualize cluster separation)
from sklearn.manifold import TSNE
emb = all_embeddings(ema_net, ALIGNED)
X, y = [], []
for name, es in emb.items():
    for e in es:
        X.append(e); y.append(name)
X = np.array(X)
if len(X) > 1:
    perp = min(30, len(X)-1)
    Z = TSNE(n_components=2, perplexity=perp, random_state=Cfg.seed,
             init="pca", learning_rate="auto").fit_transform(X)
    plt.figure(figsize=(7,6))
    for name in sorted(set(y)):
        idx = [i for i,yy in enumerate(y) if yy==name]
        plt.scatter(Z[idx,0], Z[idx,1], label=name, s=60)
    plt.legend(fontsize=8, loc="best"); plt.title("Embedding t-SNE"); plt.show()


In [ ]:
# Cell 15 — SAVE artifacts: model checkpoint + calibrated threshold
OUT = Path(Cfg.output_root); OUT.mkdir(parents=True, exist_ok=True)

ckpt_path = OUT / "siamese_backbone.pth"
# Save EMA weights (best). Also store a couple of convenience fields.
torch.save({
    "state_dict": best_state,
    "embedding_dim": Cfg.embedding_dim,
    "img_size": Cfg.img_size,
    "triplet_margin": Cfg.triplet_margin,
    "best_f1": float(best_f1),
    "best_epoch": int(best_epoch),
    "early_stopped": bool(stopped_early),
    "num_identities": len(ALIGNED),
    "architecture": "EmbeddingNet (ResNet-18 style + SE, from scratch)",
}, ckpt_path)
print("Saved checkpoint ->", ckpt_path)

# Threshold: best epoch's LOO-calibrated value (already set); all-data as backup.
final_thr = best_thr if m_val else (m_all["threshold"] if m_all else best_thr)
thr_path = OUT / "threshold.json"
with open(thr_path, "w") as f:
    json.dump({
        "cosine_distance_threshold": float(final_thr),
        "identify_threshold": float(final_thr),
        "verify_threshold": float(final_thr),
        "note": "Lower distance = more similar. Identify/verify returns 'unknown' "
                "if the best match's cosine distance exceeds this threshold.",
        "embedding_dim": Cfg.embedding_dim,
        "best_f1": float(best_f1),
        "best_epoch": int(best_epoch),
        "early_stopped": bool(stopped_early),
        "num_identities": len(ALIGNED),
        "architecture": "EmbeddingNet (ResNet-18 style + SE, from scratch)",
    }, f, indent=2)
print("Saved threshold ->", thr_path, " value=", final_thr,
      "| best_f1=", round(best_f1, 4), "@ epoch", best_epoch,
      "(early_stopped" if stopped_early else "(ran full", ")", )

print("\nDONE. Download both files from the Output panel (right sidebar -> /kaggle/working)")
print("Place them in your project's models/ folder, then run:  python app/app.py")
